# 🩺 Skin Disease Classification with ResNet50

**CNN Architecture:** ResNet50 (Transfer Learning from ImageNet)  
**Dataset:** HAM10000 — Human Against Machine with 10,000 skin lesion images  
**Task:** Multi-class skin lesion classification (7 classes)

---

## Table of Contents
1. [Part 1: Dataset Preparation](#part1)
2. [Part 2: Data Preprocessing & Augmentation](#part2)
3. [Part 3: CNN Model Development](#part3)
4. [Part 4: Model Evaluation](#part4)
5. [Part 5: ONNX Export & Demo Integration](#part5)

---

### How to Use This Notebook
1. In Colab: **Runtime → Change runtime type → T4 GPU**
2. Run cells **top to bottom** — each part depends on the previous
3. Training (Part 3) takes ~15–25 minutes on a Colab T4 GPU

---
# Part 1: Dataset Preparation (20%)
<a id='part1'></a>

**Objectives:**
- Download the HAM10000 dataset from Kaggle
- Parse the metadata CSV to get image labels
- Split into 70% train / 15% validation / 15% test (stratified)
- Display class distribution and sample images

In [ ]:
# Install required packages
!pip install -q kaggle scikit-learn matplotlib seaborn Pillow torch torchvision tqdm onnx onnxruntime

In [ ]:
# ─── Kaggle API Setup ───────────────────────────────────────────────────────
# 1. Go to https://www.kaggle.com → Account → API → Create New Token
# 2. This downloads kaggle.json — upload it when prompted below

import os
from google.colab import files

print("Upload your kaggle.json file:")
uploaded = files.upload()  # Select kaggle.json from your computer

os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
os.rename("kaggle.json", os.path.expanduser("~/.kaggle/kaggle.json"))
os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
print("✓ Kaggle API configured")

In [ ]:
# ─── Download HAM10000 Dataset ───────────────────────────────────────────────
!kaggle datasets download -d kmader/skin-cancer-mnist-ham10000 --unzip -p /content/ham10000
print("✓ Dataset downloaded")

import os
# List top-level contents to find the metadata CSV
for root, dirs, files in os.walk("/content/ham10000"):
    level = root.replace("/content/ham10000", "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    if level < 2:
        for f in files[:5]:
            print(f"{indent}  {f}")
    break

In [ ]:
import pandas as pd
import glob
import shutil
from pathlib import Path

# ─── Locate metadata CSV ─────────────────────────────────────────────────────
csv_files = glob.glob("/content/ham10000/**/*.csv", recursive=True)
print("Found CSVs:", csv_files)

# Load the metadata (try common filenames)
meta_path = next(
    (p for p in csv_files if "metadata" in p.lower() or "hamnist" in p.lower()),
    csv_files[0]
)
df = pd.read_csv(meta_path)
print(f"\nLoaded: {meta_path}")
print(f"Shape: {df.shape}")
df.head()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ─── Dataset Description ─────────────────────────────────────────────────────
CLASS_NAMES = {
    "mel":   "Melanoma",
    "nv":    "Melanocytic Nevi",
    "bcc":   "Basal Cell Carcinoma",
    "akiec": "Actinic Keratosis",
    "bkl":   "Benign Keratosis",
    "df":    "Dermatofibroma",
    "vasc":  "Vascular Lesions",
}

counts = df["dx"].value_counts()
print("=" * 55)
print(f"{'HAM10000 Dataset Description':^55}")
print("=" * 55)
print(f"{'Code':<8} {'Full Name':<28} {'Count':>6} {'%':>6}")
print("-" * 55)
for code, count in counts.items():
    pct = count / len(df) * 100
    print(f"{code:<8} {CLASS_NAMES.get(code, code):<28} {count:>6} {pct:>5.1f}%")
print("-" * 55)
print(f"{'TOTAL':<8} {'':28} {len(df):>6} {'100.0%':>6}")
print("=" * 55)

# Class distribution bar chart
fig, ax = plt.subplots(figsize=(10, 5))
colors = ["#e74c3c", "#3498db", "#2ecc71", "#f39c12", "#9b59b6", "#1abc9c", "#e67e22"]
bars = ax.bar(
    [CLASS_NAMES[c] for c in counts.index],
    counts.values,
    color=colors
)
ax.set_title("HAM10000 — Images per Class", fontsize=14, fontweight="bold")
ax.set_ylabel("Number of Images")
ax.set_xticklabels([CLASS_NAMES[c] for c in counts.index], rotation=30, ha="right")
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
            str(val), ha="center", va="bottom", fontsize=10)
plt.tight_layout()
plt.savefig("class_distribution.png", dpi=150)
plt.show()
print("\n✓ Class distribution chart saved")

In [ ]:
# ─── Locate all .jpg images ───────────────────────────────────────────────────
all_jpgs = glob.glob("/content/ham10000/**/*.jpg", recursive=True)
print(f"Total .jpg images found: {len(all_jpgs)}")

# Build image_id → file_path lookup
image_lookup = {Path(p).stem: p for p in all_jpgs}
print(f"Unique image IDs: {len(image_lookup)}")

# Match metadata rows to found images
df["filepath"] = df["image_id"].map(image_lookup)
missing = df["filepath"].isna().sum()
print(f"Missing images: {missing}")
df = df.dropna(subset=["filepath"]).reset_index(drop=True)
print(f"Usable rows: {len(df)}")

In [ ]:
from sklearn.model_selection import train_test_split
import shutil

# ─── Stratified 70/15/15 Split ────────────────────────────────────────────────
train_df, temp_df = train_test_split(
    df, test_size=0.30, stratify=df["dx"], random_state=42
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df["dx"], random_state=42
)

print(f"Train: {len(train_df)} images ({len(train_df)/len(df)*100:.1f}%)")
print(f"Val:   {len(val_df)} images ({len(val_df)/len(df)*100:.1f}%)")
print(f"Test:  {len(test_df)} images ({len(test_df)/len(df)*100:.1f}%)")

# ─── Copy into class-organized folders ──────────────────────────────────────
BASE_DIR = Path("/content/skin_data")

def copy_split(split_df, split_name):
    count = 0
    for _, row in split_df.iterrows():
        dest_dir = BASE_DIR / split_name / row["dx"]
        dest_dir.mkdir(parents=True, exist_ok=True)
        shutil.copy2(row["filepath"], dest_dir / Path(row["filepath"]).name)
        count += 1
    print(f"  {split_name}: {count} images copied")

print("\nCopying images to organized folders...")
copy_split(train_df, "train")
copy_split(val_df, "val")
copy_split(test_df, "test")
print("✓ Dataset organized into train/val/test folders")

In [ ]:
from PIL import Image
import numpy as np

# ─── Sample Images Grid ───────────────────────────────────────────────────────
print("Sample images from each class (from training set):")
fig, axes = plt.subplots(1, 7, figsize=(18, 4))
fig.suptitle("HAM10000 — One Sample Per Class", fontsize=14, fontweight="bold")

classes = sorted(df["dx"].unique())
for ax, cls in zip(axes, classes):
    sample = train_df[train_df["dx"] == cls].iloc[0]
    img = Image.open(sample["filepath"]).convert("RGB")
    ax.imshow(img)
    ax.set_title(f"{CLASS_NAMES[cls]}\n({cls})", fontsize=9, fontweight="bold")
    ax.axis("off")

plt.tight_layout()
plt.savefig("sample_images.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Sample images displayed")

### Part 1 Summary

| Split | Images | Percentage |
|-------|--------|------------|
| Train | ~7,010 | 70% |
| Validation | ~1,502 | 15% |
| Test | ~1,503 | 15% |
| **Total** | **~10,015** | **100%** |

**Dataset notes:**
- The dataset is **imbalanced** (nv = 6,705 vs vasc = 142)
- We use **stratified splits** so each class ratio is preserved across train/val/test
- Class imbalance is handled in Part 3 via **weighted CrossEntropyLoss**

---
# Part 2: Data Preprocessing & Augmentation (25%)
<a id='part2'></a>

**Objectives:**
- Define image transforms for train, validation, and test sets
- Apply data augmentation to increase effective training data size
- Create PyTorch DataLoaders
- Visualize sample augmented images

In [ ]:
import torch
import torchvision
from torchvision import transforms, datasets
from torch.utils.data import DataLoader

print(f"PyTorch version: {torch.__version__}")
print(f"Torchvision version: {torchvision.__version__}")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ─── ImageNet normalization constants ─────────────────────────────────────────
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
INPUT_SIZE    = 224

# ─── Training Transforms (with augmentation) ─────────────────────────────────
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(INPUT_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(degrees=20),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2,
        hue=0.05
    ),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# ─── Validation / Test Transforms (NO augmentation) ──────────────────────────
eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(INPUT_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

print("Training transforms:")
print(train_transform)
print("\nEval transforms:")
print(eval_transform)

In [ ]:
# ─── Create Datasets ──────────────────────────────────────────────────────────
train_dataset = datasets.ImageFolder(
    root=str(BASE_DIR / "train"),
    transform=train_transform
)
val_dataset = datasets.ImageFolder(
    root=str(BASE_DIR / "val"),
    transform=eval_transform
)
test_dataset = datasets.ImageFolder(
    root=str(BASE_DIR / "test"),
    transform=eval_transform
)

# Class names (sorted alphabetically by ImageFolder)
CLASS_ORDER = train_dataset.classes  # e.g. ['akiec', 'bcc', 'bkl', ...]
FULL_NAMES  = [CLASS_NAMES[c] for c in CLASS_ORDER]
NUM_CLASSES = len(CLASS_ORDER)

print(f"Number of classes: {NUM_CLASSES}")
print(f"Class order: {CLASS_ORDER}")
print(f"\nTrain: {len(train_dataset)} images")
print(f"Val:   {len(val_dataset)} images")
print(f"Test:  {len(test_dataset)} images")

In [ ]:
BATCH_SIZE = 32
NUM_WORKERS = 2  # Colab usually works well with 2

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True if DEVICE.type == "cuda" else False,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True if DEVICE.type == "cuda" else False,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")
print(f"Test batches:  {len(test_loader)}")
print(f"Batch size:    {BATCH_SIZE}")

In [ ]:
# ─── Preprocessing Workflow Diagram ──────────────────────────────────────────
print("Preprocessing Workflow:")
print("  RAW IMAGE (variable size)")
print("       ↓ Resize to 256×256")
print("       ↓ [Train only] RandomCrop(224)")
print("       ↓ [Train only] RandomHorizontalFlip")
print("       ↓ [Train only] RandomVerticalFlip")
print("       ↓ [Train only] RandomRotation(20°)")
print("       ↓ [Train only] ColorJitter")
print("       ↓ [Val/Test]   CenterCrop(224)")
print("       ↓ ToTensor → float32 in [0,1]")
print("       ↓ Normalize with ImageNet mean/std")
print("  OUTPUT: tensor [3, 224, 224]")


# ─── Visualize augmented images (same image, 8 augmentations) ─────────────────
def denormalize(tensor):
    """Reverse ImageNet normalization for display."""
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std  = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return torch.clamp(tensor * std + mean, 0, 1)

# Pick one sample image from the first class
sample_path = next((BASE_DIR / "train" / CLASS_ORDER[0]).iterdir())
original = Image.open(sample_path).convert("RGB")

aug_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(INPUT_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(degrees=20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
    transforms.ToTensor(),
])

fig, axes = plt.subplots(2, 5, figsize=(16, 7))
fig.suptitle(f"Data Augmentation Examples\n(Class: {CLASS_NAMES[CLASS_ORDER[0]]})",
             fontsize=13, fontweight="bold")

# Original
axes[0][0].imshow(original.resize((224, 224)))
axes[0][0].set_title("Original", fontweight="bold")
axes[0][0].axis("off")

for i, ax in enumerate(axes.flat[1:]):
    aug_img = aug_transform(original).permute(1, 2, 0).numpy()
    ax.imshow(aug_img)
    ax.set_title(f"Augmented #{i+1}", fontsize=9)
    ax.axis("off")

plt.tight_layout()
plt.savefig("augmented_samples.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Augmented samples saved")

### Part 2 Summary — Preprocessing Pipeline

| Step | Training | Validation/Test |
|------|----------|-----------------|
| Resize | 256×256 | 256 (short edge) |
| Crop | RandomCrop(224) | CenterCrop(224) |
| Flip | H-flip(50%) + V-flip(30%) | ✗ |
| Rotation | ±20° | ✗ |
| ColorJitter | brightness/contrast/saturation | ✗ |
| Normalize | ImageNet mean/std | ImageNet mean/std |

**Why augmentation?**  
Augmentation artificially expands the training set, making the model more robust to real-world variations in skin lesion photos (lighting, camera angle, zoom level).

---
# Part 3: CNN Model Development (30%)
<a id='part3'></a>

**Architecture:** ResNet50 (Transfer Learning from ImageNet)  

**Strategy:**
1. Load pretrained ResNet50 weights
2. Freeze early layers (pretrained features)
3. Unfreeze `layer4` for domain-specific fine-tuning
4. Replace the final `fc` layer for 7-class output
5. Train with class weights to handle imbalance

In [ ]:
import torch.nn as nn
from torchvision import models
from torchvision.models import ResNet50_Weights

# ─── Load Pretrained ResNet50 ─────────────────────────────────────────────────
model = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)

# ─── Freeze all layers first ──────────────────────────────────────────────────
for param in model.parameters():
    param.requires_grad = False

# ─── Unfreeze layer4 for fine-tuning ─────────────────────────────────────────
for param in model.layer4.parameters():
    param.requires_grad = True

# ─── Replace the final fully-connected layer ──────────────────────────────────
in_features = model.fc.in_features  # 2048 for ResNet50
model.fc = nn.Sequential(
    nn.Linear(in_features, 512),
    nn.ReLU(inplace=True),
    nn.Dropout(p=0.4),
    nn.Linear(512, NUM_CLASSES),
)

model = model.to(DEVICE)

# ─── Count parameters ────────────────────────────────────────────────────────
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params    = total_params - trainable_params

print("=" * 45)
print(f"{'ResNet50 Model Summary':^45}")
print("=" * 45)
print(f"  Total parameters:     {total_params:>12,}")
print(f"  Trainable parameters: {trainable_params:>12,}")
print(f"  Frozen parameters:    {frozen_params:>12,}")
print("=" * 45)
print(f"\nNew fc head:")
print(model.fc)

In [ ]:
# ─── ResNet50 Architecture Diagram ───────────────────────────────────────────
print("""
╔══════════════════════════════════════════════════════════╗
║          ResNet50 Architecture for Skin Disease          ║
╠══════════════════════════════════════════════════════════╣
║  INPUT: [batch, 3, 224, 224]                             ║
║    │                                                     ║
║    ▼  Conv1 (7×7, 64 filters, stride=2) + BN + ReLU     ║
║    ▼  MaxPool (3×3, stride=2)                            ║
║    │  → [batch, 64, 56, 56]              [FROZEN]        ║
║    │                                                     ║
║    ▼  Layer1: 3 × Bottleneck (64→256)    [FROZEN]        ║
║    │  → [batch, 256, 56, 56]                             ║
║    │                                                     ║
║    ▼  Layer2: 4 × Bottleneck (128→512)   [FROZEN]        ║
║    │  → [batch, 512, 28, 28]                             ║
║    │                                                     ║
║    ▼  Layer3: 6 × Bottleneck (256→1024)  [FROZEN]        ║
║    │  → [batch, 1024, 14, 14]                            ║
║    │                                                     ║
║    ▼  Layer4: 3 × Bottleneck (512→2048)  [TRAINABLE]     ║
║    │  → [batch, 2048, 7, 7]                              ║
║    │                                                     ║
║    ▼  Global Average Pooling → [batch, 2048]             ║
║    │                                                     ║
║    ▼  ┌────────────────────────────┐                     ║
║    │  │  Custom FC Head            │  [TRAINABLE]        ║
║    │  │  Linear(2048 → 512)        │                     ║
║    │  │  ReLU                      │                     ║
║    │  │  Dropout(0.4)              │                     ║
║    │  │  Linear(512 → 7)           │                     ║
║    │  └────────────────────────────┘                     ║
║    │                                                     ║
║  OUTPUT: logits [batch, 7]                               ║
║  (softmax → class probabilities)                         ║
╚══════════════════════════════════════════════════════════╝
""")

In [ ]:
# ─── Compute Class Weights (to handle imbalance) ──────────────────────────────
from collections import Counter

# Count samples per class in training set
train_labels = [label for _, label in train_dataset.samples]
class_counts = Counter(train_labels)
total_train  = len(train_labels)

# Inverse-frequency weighting
weights = torch.tensor(
    [total_train / (NUM_CLASSES * class_counts[i]) for i in range(NUM_CLASSES)],
    dtype=torch.float32
).to(DEVICE)

print("Class weights (higher = underrepresented):")
for i, (cls, w) in enumerate(zip(CLASS_ORDER, weights.cpu())):
    print(f"  {cls:<8} {CLASS_NAMES[cls]:<28} weight={w:.3f}  count={class_counts[i]}")

In [ ]:
# ─── Hyperparameters ─────────────────────────────────────────────────────────
EPOCHS       = 20
BATCH_SIZE   = 32       # already set above
LR_BACKBONE  = 1e-4     # lower LR for pretrained layer4
LR_HEAD      = 1e-3     # higher LR for new fc head
WEIGHT_DECAY = 1e-4
PATIENCE     = 5        # early stopping patience

# ─── Optimizer with differential learning rates ───────────────────────────────
optimizer = torch.optim.AdamW([
    {"params": model.layer4.parameters(), "lr": LR_BACKBONE},
    {"params": model.fc.parameters(),     "lr": LR_HEAD},
], weight_decay=WEIGHT_DECAY)

# ─── Learning rate scheduler ──────────────────────────────────────────────────
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS, eta_min=1e-6
)

# ─── Loss function with class weights ────────────────────────────────────────
criterion = nn.CrossEntropyLoss(weight=weights)

print("Hyperparameters:")
print(f"  Optimizer:      AdamW")
print(f"  LR (backbone):  {LR_BACKBONE}")
print(f"  LR (fc head):   {LR_HEAD}")
print(f"  Weight decay:   {WEIGHT_DECAY}")
print(f"  Scheduler:      CosineAnnealingLR (T_max={EPOCHS})")
print(f"  Loss function:  CrossEntropyLoss (weighted)")
print(f"  Batch size:     {BATCH_SIZE}")
print(f"  Epochs:         {EPOCHS}")
print(f"  Early stopping: patience={PATIENCE}")

In [ ]:
from tqdm.notebook import tqdm
import time

# ─── Training Loop ────────────────────────────────────────────────────────────
history = {
    "train_loss": [], "train_acc": [],
    "val_loss":   [], "val_acc":   [],
}

best_val_acc  = 0.0
patience_ctr  = 0
best_model_path = "/content/best_resnet50_skin.pth"


def run_epoch(loader, is_train=True):
    model.train() if is_train else model.eval()
    running_loss, correct, total = 0.0, 0, 0

    ctx = torch.enable_grad() if is_train else torch.no_grad()
    with ctx:
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)

            if is_train:
                optimizer.zero_grad()

            outputs = model(images)
            loss    = criterion(outputs, labels)

            if is_train:
                loss.backward()
                optimizer.step()

            running_loss += loss.item() * images.size(0)
            _, predicted  = outputs.max(1)
            correct      += predicted.eq(labels).sum().item()
            total        += labels.size(0)

    return running_loss / total, correct / total


print(f"Starting training for {EPOCHS} epochs on {DEVICE}...")
print("=" * 70)

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()

    train_loss, train_acc = run_epoch(train_loader, is_train=True)
    val_loss,   val_acc   = run_epoch(val_loader,   is_train=False)
    scheduler.step()

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    elapsed = time.time() - t0
    marker  = " ◄ BEST" if val_acc > best_val_acc else ""

    print(
        f"Epoch [{epoch:02d}/{EPOCHS}] "
        f"| Train Loss: {train_loss:.4f}  Acc: {train_acc*100:.2f}% "
        f"| Val Loss: {val_loss:.4f}  Acc: {val_acc*100:.2f}% "
        f"| {elapsed:.0f}s{marker}"
    )

    # ── Save best model ────────────────────────────────────────────────────
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            "epoch":       epoch,
            "model_state": model.state_dict(),
            "optimizer":   optimizer.state_dict(),
            "val_acc":     best_val_acc,
            "class_order": CLASS_ORDER,
        }, best_model_path)
        patience_ctr = 0
    else:
        patience_ctr += 1
        if patience_ctr >= PATIENCE:
            print(f"\nEarly stopping triggered at epoch {epoch} (patience={PATIENCE})")
            break

print("=" * 70)
print(f"\n✓ Training complete! Best validation accuracy: {best_val_acc*100:.2f}%")
print(f"  Best model saved to: {best_model_path}")

In [ ]:
# ─── Load the best saved model ────────────────────────────────────────────────
checkpoint = torch.load(best_model_path, map_location=DEVICE)
model.load_state_dict(checkpoint["model_state"])
print(f"✓ Loaded best model from epoch {checkpoint['epoch']}")
print(f"  Best validation accuracy: {checkpoint['val_acc']*100:.2f}%")

### Part 3 Summary — Hyperparameters Used

| Hyperparameter | Value | Reason |
|----------------|-------|--------|
| Architecture | ResNet50 | Assigned architecture; 50-layer residual network |
| Pretrained weights | ImageNet (IMAGENET1K_V2) | Transfer learning baseline |
| Frozen layers | conv1, layer1, layer2, layer3 | Preserve general features |
| Fine-tuned layers | layer4 + custom fc head | Adapt to skin lesion domain |
| Optimizer | AdamW | Adaptive LR with weight decay |
| LR (backbone) | 1e-4 | Gentle updates to pretrained weights |
| LR (fc head) | 1e-3 | Faster learning for new classifier |
| Weight decay | 1e-4 | Regularization |
| Scheduler | CosineAnnealingLR | Smooth LR warmdown |
| Loss function | CrossEntropyLoss (weighted) | Handles class imbalance |
| Batch size | 32 | Fits in Colab T4 16GB |
| Epochs | 20 (+ early stopping) | Sufficient for convergence |

---
# Part 4: Model Evaluation (25%)
<a id='part4'></a>

**Objectives:**
- Compute Accuracy, Precision, Recall, F1-Score on the **test set**
- Plot Training/Validation Accuracy and Loss curves
- Generate a Confusion Matrix heatmap
- Show 20 sample predictions with confidence scores

In [ ]:
# ─── Training & Validation Curves ────────────────────────────────────────────
epochs_ran = len(history["train_loss"])
epoch_range = range(1, epochs_ran + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Training History", fontsize=14, fontweight="bold")

# Accuracy
axes[0].plot(epoch_range, [a*100 for a in history["train_acc"]], "b-o", label="Train Acc", markersize=4)
axes[0].plot(epoch_range, [a*100 for a in history["val_acc"]],   "r-o", label="Val Acc",   markersize=4)
axes[0].set_title("Accuracy", fontsize=12)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy (%)")
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(0, 100)

# Loss
axes[1].plot(epoch_range, history["train_loss"], "b-o", label="Train Loss", markersize=4)
axes[1].plot(epoch_range, history["val_loss"],   "r-o", label="Val Loss",   markersize=4)
axes[1].set_title("Loss", fontsize=12)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("training_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"✓ Training curves saved | Final Val Acc: {history['val_acc'][-1]*100:.2f}%")

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)
import seaborn as sns

# ─── Evaluate on Test Set ────────────────────────────────────────────────────
model.eval()
all_preds, all_labels, all_probs = [], [], []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(DEVICE)
        outputs = model(images)
        probs   = torch.softmax(outputs, dim=1)
        _, preds = probs.max(1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())
        all_probs.extend(probs.cpu().numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

# ─── Performance Metrics Table ────────────────────────────────────────────────
acc = accuracy_score(all_labels, all_preds)
prec_macro = precision_score(all_labels, all_preds, average="macro", zero_division=0)
rec_macro  = recall_score(all_labels,   all_preds, average="macro", zero_division=0)
f1_macro   = f1_score(all_labels,       all_preds, average="macro", zero_division=0)

print("=" * 60)
print(f"{'TEST SET PERFORMANCE METRICS':^60}")
print("=" * 60)
print(f"  Overall Accuracy:       {acc*100:.2f}%")
print(f"  Macro Precision:        {prec_macro*100:.2f}%")
print(f"  Macro Recall:           {rec_macro*100:.2f}%")
print(f"  Macro F1-Score:         {f1_macro*100:.2f}%")
print("=" * 60)

print("\nPer-class metrics:")
print(classification_report(
    all_labels, all_preds,
    target_names=FULL_NAMES,
    zero_division=0
))

In [ ]:
# ─── Per-class Metrics DataFrame ─────────────────────────────────────────────
prec_per = precision_score(all_labels, all_preds, average=None, zero_division=0)
rec_per  = recall_score(all_labels,   all_preds, average=None, zero_division=0)
f1_per   = f1_score(all_labels,       all_preds, average=None, zero_division=0)

metrics_df = pd.DataFrame({
    "Class":     FULL_NAMES,
    "Precision": [f"{p*100:.1f}%" for p in prec_per],
    "Recall":    [f"{r*100:.1f}%" for r in rec_per],
    "F1-Score":  [f"{f*100:.1f}%" for f in f1_per],
})

# Add totals row
totals = pd.DataFrame([{
    "Class":     "MACRO AVERAGE",
    "Precision": f"{prec_macro*100:.1f}%",
    "Recall":    f"{rec_macro*100:.1f}%",
    "F1-Score":  f"{f1_macro*100:.1f}%",
}])
metrics_df = pd.concat([metrics_df, totals], ignore_index=True)

print("Performance Metrics Table:")
print(metrics_df.to_string(index=False))

In [ ]:
# ─── Confusion Matrix ────────────────────────────────────────────────────────
cm = confusion_matrix(all_labels, all_preds)

# Normalize to percentages
cm_normalized = cm.astype("float") / cm.sum(axis=1, keepdims=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Raw counts
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=FULL_NAMES, yticklabels=FULL_NAMES, ax=axes[0]
)
axes[0].set_title("Confusion Matrix (Counts)", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Predicted Label", fontsize=11)
axes[0].set_ylabel("True Label", fontsize=11)
axes[0].set_xticklabels(FULL_NAMES, rotation=30, ha="right")
axes[0].set_yticklabels(FULL_NAMES, rotation=0)

# Normalized percentages
sns.heatmap(
    cm_normalized, annot=True, fmt=".1f", cmap="Blues",
    xticklabels=FULL_NAMES, yticklabels=FULL_NAMES, ax=axes[1]
)
axes[1].set_title("Confusion Matrix (% per True Class)", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Predicted Label", fontsize=11)
axes[1].set_ylabel("True Label", fontsize=11)
axes[1].set_xticklabels(FULL_NAMES, rotation=30, ha="right")
axes[1].set_yticklabels(FULL_NAMES, rotation=0)

plt.suptitle("ResNet50 — Skin Disease Classification", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Confusion matrix saved")

In [ ]:
# ─── 20 Sample Predictions ────────────────────────────────────────────────────
model.eval()
sample_indices = np.random.choice(len(test_dataset), 20, replace=False)

fig, axes = plt.subplots(4, 5, figsize=(20, 16))
fig.suptitle("20 Sample Predictions from Test Set", fontsize=16, fontweight="bold")

for trial_num, (ax, idx) in enumerate(zip(axes.flat, sample_indices)):
    img_tensor, true_label = test_dataset[idx]

    # Run inference
    with torch.no_grad():
        output = model(img_tensor.unsqueeze(0).to(DEVICE))
        probs  = torch.softmax(output, dim=1).squeeze().cpu().numpy()
        pred_label    = int(probs.argmax())
        confidence    = float(probs.max())

    # Denormalize for display
    mean = np.array(IMAGENET_MEAN).reshape(3, 1, 1)
    std  = np.array(IMAGENET_STD).reshape(3, 1, 1)
    img_display = np.clip(img_tensor.numpy() * std + mean, 0, 1).transpose(1, 2, 0)

    is_correct = pred_label == true_label
    border_color = "green" if is_correct else "red"

    ax.imshow(img_display)
    ax.set_title(
        f"Trial #{trial_num+1}\n"
        f"True: {CLASS_ORDER[true_label]}\n"
        f"Pred: {CLASS_ORDER[pred_label]} ({confidence*100:.1f}%)",
        fontsize=8,
        color=border_color,
        fontweight="bold"
    )
    for spine in ax.spines.values():
        spine.set_edgecolor(border_color)
        spine.set_linewidth(3)
    ax.axis("off")

correct_count = sum(
    int(np.argmax(torch.softmax(model(test_dataset[i][0].unsqueeze(0).to(DEVICE)), dim=1).squeeze().cpu().numpy())) == test_dataset[i][1]
    for i in sample_indices
)
plt.figtext(0.5, 0.01,
    f"Correct: {correct_count}/20 | Green border = correct, Red border = incorrect",
    ha="center", fontsize=11
)
plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.savefig("sample_predictions.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"✓ 20-trial predictions saved | Correct: {correct_count}/20")

### Part 4 Summary

All four visualizations have been saved:
- `training_curves.png` — Training/Validation Accuracy and Loss graphs
- `confusion_matrix.png` — Confusion matrix (raw counts + normalized)
- `sample_predictions.png` — 20 trial predictions (green = correct, red = wrong)

**Reading the confusion matrix:**  
- Diagonal cells = correct predictions  
- Off-diagonal = misclassifications  
- Classes with similar visual appearance (e.g., nv ↔ mel) will have more confusion

---
# Part 5: ONNX Export & Demo Integration
<a id='part5'></a>

**Objectives:**
- Export the trained PyTorch model to ONNX format
- Generate `skin_labels.json` for the Next.js demo app
- Verify the ONNX export produces identical predictions
- Download files and plug into the existing web app

In [ ]:
import onnx
import onnxruntime as ort

ONNX_PATH = "/content/resnet50_skin.onnx"

# ─── Export to ONNX ──────────────────────────────────────────────────────────
model.eval()
dummy_input = torch.randn(1, 3, 224, 224).to(DEVICE)

torch.onnx.export(
    model,
    dummy_input,
    ONNX_PATH,
    export_params=True,
    opset_version=14,
    do_constant_folding=True,
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={
        "input":  {0: "batch_size"},
        "output": {0: "batch_size"},
    },
    verbose=False,
)

# ─── Verify ONNX model ────────────────────────────────────────────────────────
onnx_model = onnx.load(ONNX_PATH)
onnx.checker.check_model(onnx_model)

import os
size_mb = os.path.getsize(ONNX_PATH) / 1e6
print(f"✓ ONNX model exported and verified")
print(f"  Path:  {ONNX_PATH}")
print(f"  Size:  {size_mb:.1f} MB")
print(f"  Opset: 14")

In [ ]:
# ─── Verify ONNX output matches PyTorch ──────────────────────────────────────
ort_session = ort.InferenceSession(ONNX_PATH, providers=["CPUExecutionProvider"])

# Use a real test image
test_img, test_label = test_dataset[0]
test_input = test_img.unsqueeze(0).numpy()  # shape: [1, 3, 224, 224]

# PyTorch inference
with torch.no_grad():
    pt_output = model(torch.from_numpy(test_input).to(DEVICE)).cpu().numpy()

# ONNX Runtime inference
ort_output = ort_session.run(["output"], {"input": test_input})[0]

max_diff = np.abs(pt_output - ort_output).max()
print(f"Max absolute difference (PyTorch vs ONNX): {max_diff:.6f}")
assert max_diff < 1e-4, "Outputs differ too much!"
print("✓ ONNX outputs match PyTorch (max diff < 1e-4)")

# Show prediction
pt_pred_class = CLASS_ORDER[pt_output.argmax()]
true_class    = CLASS_ORDER[test_label]
confidence    = float(np.exp(pt_output.max()) / np.exp(pt_output).sum() * 100)
print(f"\nSample prediction:")
print(f"  True class:      {true_class} ({CLASS_NAMES[true_class]})")
print(f"  Predicted class: {pt_pred_class} ({CLASS_NAMES[pt_pred_class]})")

In [ ]:
import json

# ─── Generate skin_labels.json ────────────────────────────────────────────────
# This replaces imagenet_labels.json in the Next.js app.
# Format: array of strings, index = class index used by the model.
LABELS_PATH = "/content/skin_labels.json"

skin_labels = FULL_NAMES  # e.g. ["Actinic Keratosis", "Basal Cell Carcinoma", ...]

with open(LABELS_PATH, "w") as f:
    json.dump(skin_labels, f, indent=2)

print("✓ skin_labels.json generated:")
print(json.dumps(skin_labels, indent=2))

In [ ]:
# ─── Download files to your computer ─────────────────────────────────────────
from google.colab import files

print("Downloading resnet50_skin.onnx (~100 MB)...")
files.download(ONNX_PATH)

print("Downloading skin_labels.json...")
files.download(LABELS_PATH)

print("Downloading training history plots...")
files.download("training_curves.png")
files.download("confusion_matrix.png")
files.download("sample_predictions.png")
files.download("class_distribution.png")
files.download("augmented_samples.png")
files.download("sample_images.png")

print("\n✓ All files downloaded!")

In [ ]:
# ─── (Optional) Save to Google Drive ─────────────────────────────────────────
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

drive_dir = Path("/content/drive/MyDrive/skin_classifier")
drive_dir.mkdir(parents=True, exist_ok=True)

shutil.copy(ONNX_PATH, drive_dir / "resnet50_skin.onnx")
shutil.copy(LABELS_PATH, drive_dir / "skin_labels.json")
shutil.copy(best_model_path, drive_dir / "best_resnet50_skin.pth")
for png in ["training_curves.png", "confusion_matrix.png",
            "sample_predictions.png", "class_distribution.png",
            "augmented_samples.png", "sample_images.png"]:
    if Path(png).exists():
        shutil.copy(png, drive_dir / png)

print(f"✓ All files backed up to Google Drive: {drive_dir}")

### Part 5 — Demo Integration Guide

After downloading the files, plug them into the existing Next.js web app:

```
py-react-onnx-resnet50/
├── public/
│   ├── models/
│   │   └── resnet50_skin.onnx    ← copy here
│   └── data/
│       └── skin_labels.json      ← copy here
```

Then open `app/components/ImageClassifier.js` and change two lines:

```js
// Before:
const MODEL_PATH  = "/models/resnet50.onnx";
const LABELS_PATH = "/data/imagenet_labels.json";

// After:
const MODEL_PATH  = "/models/resnet50_skin.onnx";
const LABELS_PATH = "/data/skin_labels.json";
```

Start the app:
```bash
npm run dev
```

Open `http://localhost:3000` → Upload a skin lesion image → Click **Classify image** → See the predicted skin condition with confidence score!

---

### Demo Checklist
- [ ] Upload an image
- [ ] Show the predicted class (e.g., "Melanoma")
- [ ] Display confidence score (e.g., "94.3%")
- [ ] Explain the ResNet50 architecture using the diagram from Part 3
- [ ] Walk through the training curves and confusion matrix

In [ ]:
# ─── Final Project Summary ────────────────────────────────────────────────────
print("=" * 65)
print(f"{'SKIN DISEASE CLASSIFICATION — PROJECT SUMMARY':^65}")
print("=" * 65)
print()
print("PART 1: Dataset Preparation")
print(f"  Dataset:        HAM10000")
print(f"  Total images:   {len(df):,}")
print(f"  Classes:        {NUM_CLASSES} ({', '.join(CLASS_ORDER)})")
print(f"  Train/Val/Test: 70% / 15% / 15% (stratified)")
print()
print("PART 2: Preprocessing")
print(f"  Input size:     224×224")
print(f"  Normalization:  ImageNet mean/std")
print(f"  Augmentations:  Flip, Rotation, ColorJitter")
print()
print("PART 3: Model")
print(f"  Architecture:   ResNet50 (Transfer Learning)")
print(f"  Pretrained:     ImageNet (IMAGENET1K_V2)")
print(f"  Fine-tuned:     layer4 + custom fc head")
print(f"  Optimizer:      AdamW (lr=1e-4/1e-3)")
print(f"  Epochs trained: {len(history['train_loss'])}")
print(f"  Best Val Acc:   {best_val_acc*100:.2f}%")
print()
print("PART 4: Evaluation")
print(f"  Test Accuracy:  {acc*100:.2f}%")
print(f"  Macro F1:       {f1_macro*100:.2f}%")
print(f"  Macro Precision:{prec_macro*100:.2f}%")
print(f"  Macro Recall:   {rec_macro*100:.2f}%")
print()
print("PART 5: Demo")
print(f"  ONNX model:     resnet50_skin.onnx")
print(f"  Labels file:    skin_labels.json")
print(f"  Demo:           Next.js browser app (localhost:3000)")
print("=" * 65)